# 04 BM25

Se implementa BM25 y se compara su ranking con TF-IDF y similitud coseno en la misma colección. Se desglosa el aporte de cada término para explicar una inversión de posiciones y el efecto de $k_1$ y $b$.

## 1. Carga y preprocesamiento del corpus

Se conservan los 100 documentos de Project Gutenberg del deber 03. Se tokeniza el TXT completo, incluidos los encabezados y la licencia, con `casefold()` y el patrón Unicode `[^\W_]+`. Se conservan letras, números y tildes. La puntuación y el guion bajo separan tokens. Se mantienen las palabras vacías y las formas originales de los términos.

La ejecución requiere Jupyter, `numpy`, `pandas`, `scipy` y `scikit-learn`. El corpus local se busca en `epub` junto al repositorio. La variable `RI_CORPUS` permite indicar otra carpeta con los TXT o un archivo ZIP/TAR.

In [1]:
from collections import Counter
from pathlib import Path, PurePosixPath
import hashlib
import os
import re
import sys
import tarfile
import zipfile

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import norm as norma_dispersa
from sklearn.metrics.pairwise import cosine_similarity

PATRON_TOKEN = re.compile(r"[^\W_]+", flags=re.UNICODE)


def tokenizar(texto):
    return PATRON_TOKEN.findall(texto.casefold())


base = Path.cwd()
raiz = next((p for p in (base, *base.parents) if (p / "Trabajos/Matrices.ipynb").is_file()), base)
ruta_configurada = os.getenv("RI_CORPUS")
ruta_archivo = (Path(os.getenv("LOCALAPPDATA", str(Path.home() / ".local")))
                / "GutenbergRI/txt-files.tar.zip")
rutas = ([Path(ruta_configurada).expanduser()] if ruta_configurada else
         [raiz.parent / "epub", raiz / "epub", raiz / "gutenberg_grande", ruta_archivo])
ruta_corpus = next((p for p in rutas if p.exists()), None)
if ruta_corpus is None:
    raise FileNotFoundError("Se requiere el corpus original en epub o en la ruta indicada por RI_CORPUS.")


IDS_ORIGINALES = """
1 2 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 100 101 102
103 104 105 106 107 108 109 110 111 112 113 115 117 118 119 120 121 122 123 124
125 126 128 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146
147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166
167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 200 201 202 203 204
""".split()

IDS_ELEGIDOS = set(IDS_ORIGINALES)

def identificar_documento(nombre):
    archivo = PurePosixPath(str(nombre).replace("\\", "/")).name
    coincidencia = re.fullmatch(r"pg(\d+)\.txt", archivo, flags=re.IGNORECASE)
    return coincidencia[1] if coincidencia and coincidencia[1] in IDS_ELEGIDOS else None

def leer_tar(flujo):
    with tarfile.open(fileobj=flujo, mode="r|*") as archivo_tar:
        for miembro in archivo_tar:
            if miembro.isfile() and identificar_documento(miembro.name):
                with archivo_tar.extractfile(miembro) as documento:
                    yield miembro.name, documento.read()

def iterar_documentos(ruta):
    if ruta.is_dir():
        archivos = sorted(ruta.rglob("*.txt"), key=lambda p: (
            int(p.parent.name) if p.parent.name.isdigit() else float("inf"),
            str(p.relative_to(ruta)).casefold()))
        for archivo in archivos:
            if identificar_documento(archivo.name):
                yield str(archivo.relative_to(ruta)), archivo.read_bytes()
    elif zipfile.is_zipfile(ruta):
        with zipfile.ZipFile(ruta) as archivo_zip:
            for miembro in sorted(archivo_zip.infolist(), key=lambda m: m.filename):
                if miembro.filename.lower().endswith((".tar", ".tar.gz", ".tgz")):
                    with archivo_zip.open(miembro) as flujo:
                        yield from leer_tar(flujo)
                elif not miembro.is_dir() and identificar_documento(miembro.filename):
                    yield miembro.filename, archivo_zip.read(miembro)
    else:
        with ruta.open("rb") as flujo:
            yield from leer_tar(flujo)

def leer_texto(contenido):
    try:
        texto = contenido.decode("utf-8-sig")
        codificacion = "utf-8-sig"
    except UnicodeDecodeError:
        texto = contenido.decode("cp1252")
        codificacion = "cp1252"
    return texto.replace("\r\n", "\n").replace("\r", "\n"), codificacion

def obtener_titulo(texto, doc_id):
    encabezado = re.split(r"\*{3}\s*START OF", texto[:20000], maxsplit=1, flags=re.I)[0]
    patron = r"^Title:[ \t]*([^\n]*(?:\n(?!\s*\n|[ \t]*[A-Za-z][A-Za-z -]*:|\*{3})[^\n]+)*)"
    coincidencia = re.search(patron, encabezado, flags=re.I | re.M)
    return " ".join(coincidencia[1].split()) if coincidencia else f"Gutenberg {doc_id} (pg{doc_id}.txt)"

In [2]:
documentos_por_id = {}
huellas = set()
candidatos = iterar_documentos(ruta_corpus)
try:
    for nombre, contenido in candidatos:
        doc_id = identificar_documento(nombre)
        if doc_id in documentos_por_id:
            continue
        texto, _ = leer_texto(contenido)
        if "project gutenberg" not in texto[:5000].casefold():
            continue
        huella = hashlib.sha256(texto.encode("utf-8")).hexdigest()
        if huella in huellas:
            continue
        tokens = [sys.intern(token) for token in tokenizar(texto)]
        if not tokens:
            continue
        documentos_por_id[doc_id] = (doc_id, obtener_titulo(texto, doc_id), texto, tokens)
        huellas.add(huella)
        if len(documentos_por_id) == len(IDS_ORIGINALES):
            break
finally:
    candidatos.close()

faltantes = [doc_id for doc_id in IDS_ORIGINALES if doc_id not in documentos_por_id]
if faltantes:
    raise ValueError(f"Faltan estos documentos del corpus original: {faltantes}")

corpus = pd.DataFrame([documentos_por_id[doc_id] for doc_id in IDS_ORIGINALES],
                      columns=["Identificador", "Título", "Texto", "tokens"])
ids = corpus["Identificador"].tolist()
n_documentos = len(corpus)
longitudes = corpus["tokens"].map(len).to_numpy()
longitud_media = longitudes.mean()
with pd.option_context("display.max_colwidth", 75):
    display(corpus.head())
print(f"Documentos: {n_documentos} | Longitud media: {longitud_media:.2f} tokens")

,Identificador,Título,Texto,tokens
0,1,The Declaration of Independence of the United States of America,The Project Gutenberg eBook of The Declaration of Independence of the U...,"[the, project, gutenberg, ebook, of, the, declaration, of, independence..."
1,2,The United States Bill of Rights,The Project Gutenberg eBook of The United States Bill of Rights\n \n...,"[the, project, gutenberg, ebook, of, the, united, states, bill, of, rig..."
2,10,The King James Version of the Bible,The Project Gutenberg eBook of The King James Version of the Bible\n ...,"[the, project, gutenberg, ebook, of, the, king, james, version, of, the..."
3,11,Alice's Adventures in Wonderland,The Project Gutenberg eBook of Alice's Adventures in Wonderland\n \n...,"[the, project, gutenberg, ebook, of, alice, s, adventures, in, wonderla..."
4,12,Through the Looking-Glass,The Project Gutenberg eBook of Through the Looking-Glass\n \nThis eB...,"[the, project, gutenberg, ebook, of, through, the, looking, glass, this..."


Documentos: 100 | Longitud media: 131569.15 tokens


Para cada término, $df(t)$ cuenta los documentos que lo contienen. Con $N$ documentos se calculan los dos IDF sobre el mismo vocabulario.

$$\operatorname{IDF}_{\mathrm{TFIDF}}(t)=1+\ln\left(\frac{N}{df(t)}\right),\qquad
\operatorname{IDF}_{\mathrm{BM25}}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right).$$

Se conservan las matrices de frecuencias y de TF-IDF en formato disperso. La tabla de términos muestra una muestra de las estadísticas calculadas para todo el vocabulario.

In [3]:
def construir_matriz_tf(frecuencias_documentos, columnas):
    filas, indices, valores = [], [], []
    for fila, frecuencias in enumerate(frecuencias_documentos):
        for termino, frecuencia in frecuencias.items():
            if termino in columnas:
                filas.append(fila)
                indices.append(columnas[termino])
                valores.append(frecuencia)
    matriz = csr_matrix((valores, (filas, indices)),
                        shape=(len(frecuencias_documentos), len(columnas)), dtype=np.int64)
    matriz.sort_indices()
    return matriz


contadores = [Counter(tokens) for tokens in corpus["tokens"]]
df_por_termino = Counter(termino for frecuencias in contadores for termino in frecuencias)
vocabulario = sorted(df_por_termino)
columnas = {termino: i for i, termino in enumerate(vocabulario)}
df = np.array([df_por_termino[t] for t in vocabulario])
idf_tfidf = np.log(n_documentos / df) + 1
idf_bm25 = np.log1p((n_documentos - df + 0.5) / (df + 0.5))
estadisticas = pd.DataFrame({"Término": vocabulario, "DF": df,
                             "IDF TF-IDF": idf_tfidf, "IDF BM25": idf_bm25}).set_index("Término")

matriz_tf = construir_matriz_tf(contadores, columnas)
matriz_tfidf = matriz_tf.multiply(idf_tfidf).tocsr()

assert n_documentos == 100 and longitudes.sum() == 13_156_915 and len(vocabulario) == 191_063
assert np.array_equal(df, np.asarray((matriz_tf > 0).sum(axis=0)).ravel())
print(f"Vocabulario: {len(vocabulario)} | Tokens: {longitudes.sum()} | Matriz TF-IDF: {matriz_tfidf.shape}")
display(estadisticas.loc[["the", "island", "treasure", "pirates", "jabberwock", "passepartout"]].round(6))

Vocabulario: 191063 | Tokens: 13156915 | Matriz TF-IDF: (100, 191063)


,DF,IDF TF-IDF,IDF BM25
Término,,,
the,100,1.000000,0.004963
island,61,1.494296,0.496083
treasure,53,1.634878,0.635439
pirates,19,2.660731,1.644706
jabberwock,2,4.912023,3.698830
passepartout,1,5.605170,4.209655


## 2. Benchmarking

Se seleccionan ocho consultas en inglés de una a seis palabras. `the` aparece en los 100 documentos. `jabberwock` y `bandersnatch` aparecen en dos documentos cada uno. La consulta sobre Phileas Fogg combina tres términos exclusivos de un documento con palabras que aparecen en 86 o 100 documentos.

TF-IDF usa la frecuencia absoluta en documentos y consultas y los compara mediante coseno. BM25 suma un aporte por cada término distinto del vocabulario, con $k_1=1.2$ y $b=0.75$ como configuración base.

$$\operatorname{BM25}(q,d)=\sum_{t\in q\cap V}
\operatorname{IDF}_{\mathrm{BM25}}(t)\,
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\,|d|/\overline{|d|}\right)}.$$

$f(t,d)$ es la frecuencia del término en el documento, $|d|$ es la longitud en tokens y $\overline{|d|}$ es la longitud media de la colección. Las estadísticas se calculan una sola vez. Los resultados con puntuación positiva se ordenan de mayor a menor y los empates se resuelven por ID ascendente como cadena. Cada lista contiene hasta diez documentos.

In [4]:
consultas = {
    "Q1": ("the", "Se buscan apariciones del artículo definido inglés."),
    "Q2": ("jabberwock bandersnatch", "Se buscan alusiones a dos criaturas de Lewis Carroll."),
    "Q3": ("alice rabbit wonderland", "Se busca el relato de Alicia y el conejo en Wonderland."),
    "Q4": ("pirates treasure island", "Se buscan aventuras de piratas y tesoros en una isla."),
    "Q5": ("love marriage elinor marianne", "Se buscan las relaciones amorosas de Elinor y Marianne."),
    "Q6": ("united states independence constitution", "Se buscan textos sobre la independencia y la constitución de Estados Unidos."),
    "Q7": ("sherlock holmes detective investigation", "Se buscan relatos de investigaciones de Sherlock Holmes."),
    "Q8": ("phileas fogg passepartout around the world", "Se busca la novela de la vuelta al mundo de Phileas Fogg y Passepartout."),
}
tokens_consultas = {clave: tokenizar(texto) for clave, (texto, _) in consultas.items()}
tabla_consultas = pd.DataFrame([
    {"Consulta": clave, "Texto": texto, "Necesidad de información": intencion,
     "Palabras": len(tokens_consultas[clave]),
     "DF por término": ", ".join(f"{t} ({df_por_termino[t]})" for t in tokens_consultas[clave])}
    for clave, (texto, intencion) in consultas.items()
]).set_index("Consulta")
with pd.option_context("display.max_colwidth", None):
    display(tabla_consultas)

K1_BASE, B_BASE = 1.2, 0.75
matriz_tf_columnas = matriz_tf.tocsc()


def vector_tfidf(tokens):
    tf = construir_matriz_tf([Counter(tokens)], columnas)
    return tf.multiply(idf_tfidf).tocsr()


def puntuar_coseno(tokens):
    similitud = cosine_similarity(vector_tfidf(tokens), matriz_tfidf, dense_output=False)
    return similitud.toarray().ravel()


def aportes_bm25(tokens, k1=K1_BASE, b=B_BASE):
    terminos = [t for t in dict.fromkeys(tokens) if t in columnas]
    indices = [columnas[t] for t in terminos]
    frecuencias = matriz_tf_columnas[:, indices].toarray()
    ajuste = k1 * (1 - b + b * longitudes / longitud_media)
    aportes = idf_bm25[indices] * frecuencias * (k1 + 1) / (frecuencias + ajuste[:, None])
    return terminos, frecuencias, aportes


def ordenar_positivos(puntuaciones, doc_ids):
    return sorted(np.flatnonzero(puntuaciones > 0), key=lambda i: (-puntuaciones[i], doc_ids[i]))


resultados = {}
for clave, tokens in tokens_consultas.items():
    coseno = puntuar_coseno(tokens)
    bm25 = aportes_bm25(tokens)[2].sum(axis=1)
    resultados[clave] = {
        "coseno": coseno, "bm25": bm25,
        "orden_coseno": ordenar_positivos(coseno, ids),
        "orden_bm25": ordenar_positivos(bm25, ids),
    }

tokens_ausentes = tokenizar("zzgutenbergzz zzvocabcheckzz")
assert all(t not in columnas for t in tokens_ausentes)
assert ordenar_positivos(puntuar_coseno(tokens_ausentes), ids) == []
assert ordenar_positivos(aportes_bm25(tokens_ausentes)[2].sum(axis=1), ids) == []
assert ordenar_positivos(np.array([0.5, 0.5, 0.5, 0]), ["2", "10", "1", "0"]) == [2, 1, 0]
print("La consulta sin vocabulario devuelve dos listas vacías. En empates, el ID 10 precede al ID 2.")

,Texto,Necesidad de información,Palabras,DF por término
Consulta,,,,
Q1,the,Se buscan apariciones del artículo definido inglés.,1,the (100)
Q2,jabberwock bandersnatch,Se buscan alusiones a dos criaturas de Lewis Carroll.,2,"jabberwock (2), bandersnatch (2)"
Q3,alice rabbit wonderland,Se busca el relato de Alicia y el conejo en Wonderland.,3,"alice (21), rabbit (25), wonderland (10)"
Q4,pirates treasure island,Se buscan aventuras de piratas y tesoros en una isla.,3,"pirates (19), treasure (53), island (61)"
Q5,love marriage elinor marianne,Se buscan las relaciones amorosas de Elinor y Marianne.,4,"love (88), marriage (63), elinor (2), marianne (4)"
Q6,united states independence constitution,Se buscan textos sobre la independencia y la constitución de Estados Unidos.,4,"united (100), states (100), independence (50), constitution (45)"
Q7,sherlock holmes detective investigation,Se buscan relatos de investigaciones de Sherlock Holmes.,4,"sherlock (6), holmes (10), detective (17), investigation (40)"
Q8,phileas fogg passepartout around the world,Se busca la novela de la vuelta al mundo de Phileas Fogg y Passepartout.,6,"phileas (1), fogg (1), passepartout (1), around (86), the (100), world (100)"


La consulta sin vocabulario devuelve dos listas vacías. En empates, el ID 10 precede al ID 2.


## 3. Comparación de los rankings

In [5]:
rotulos = (corpus["Identificador"] + " · " + corpus["Título"]).tolist()


def columnas_ranking(orden, puntuaciones, nombre, nombre_score):
    return pd.DataFrame({
        nombre: [rotulos[i] for i in orden],
        nombre_score: [f"{puntuaciones[i]:.10f}" for i in orden],
    }, index=pd.RangeIndex(1, len(orden) + 1, name="Posición"))


resumen = []
for clave, resultado in resultados.items():
    top_coseno = resultado["orden_coseno"][:10]
    top_bm25 = resultado["orden_bm25"][:10]
    tabla = pd.concat([
        columnas_ranking(top_coseno, resultado["coseno"], "ID y título TF-IDF", "Coseno"),
        columnas_ranking(top_bm25, resultado["bm25"], "ID y título BM25", "Score BM25"),
    ], axis=1).fillna("—")
    ids_coseno = [ids[i] for i in top_coseno]
    ids_bm25 = [ids[i] for i in top_bm25]
    resumen.append({
        "Consulta": clave, "Tamaño TF-IDF": len(top_coseno), "Tamaño BM25": len(top_bm25),
        "IDs en común": len(set(ids_coseno) & set(ids_bm25)),
        "Primer ID TF-IDF": next(iter(ids_coseno), "—"),
        "Primer ID BM25": next(iter(ids_bm25), "—"),
    })
    display(Markdown(f"**{clave} · {consultas[clave][0]}**"))
    with pd.option_context("display.max_colwidth", None):
        display(tabla)

display(pd.DataFrame(resumen).set_index("Consulta"))

**Q1 · the**

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,21 · Three hundred Aesop’s fables,0.7572894986,"200 · The Project Gutenberg Encyclopedia, Volume 1 of 28",0.0109172527
2,19 · The Song of Hiawatha,0.7412666089,18 · The Federalist Papers,0.0109171664
3,"200 · The Project Gutenberg Encyclopedia, Volume 1 of 28",0.7289267458,10 · The King James Version of the Bible,0.0109170898
4,164 · Twenty Thousand Leagues under the Sea,0.7086328728,135 · Les Misérables,0.0109170069
5,18 · The Federalist Papers,0.6999930938,"17 · The Book of Mormon an account written by the hand of Mormon, upon plates taken from the plates of Nephi",0.0109168691
6,136 · A Child's Garden of Verses,0.6923658314,164 · Twenty Thousand Leagues under the Sea,0.0109168344
7,151 · The Rime of the Ancient Mariner,0.6822871601,"15 · Moby-Dick; or, The Whale",0.0109167860
8,168 · The History and Practice of the Art of Photography,0.6807421838,21 · Three hundred Aesop’s fables,0.0109166770
9,135 · Les Misérables,0.6674865975,106 · Jungle Tales of Tarzan,0.0109166743


**Q2 · jabberwock bandersnatch**

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,13 · The Hunting of the Snark: An Agony in Eight Fits,0.0154070342,12 · Through the Looking-Glass,13.8139511974
2,12 · Through the Looking-Glass,0.0059614038,13 · The Hunting of the Snark: An Agony in Eight Fits,13.2623195265


**Q3 · alice rabbit wonderland**

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,11 · Alice's Adventures in Wonderland,0.1932280073,11 · Alice's Adventures in Wonderland,11.0411128926
2,12 · Through the Looking-Glass,0.1830560928,157 · Daddy-Long-Legs,8.3865426078
3,133 · The Damnation of Theron Ware,0.0193094670,12 · Through the Looking-Glass,6.6453105810
4,159 · The island of Doctor Moreau,0.0043318488,130 · Orthodoxy,6.4380871233
5,19 · The Song of Hiawatha,0.0027133090,152 · Wild Justice,5.9896623819
6,125 · A Girl of the Limberlost,0.0024448156,133 · The Damnation of Theron Ware,5.6917435757
7,157 · Daddy-Long-Legs,0.0020717919,204 · The innocence of Father Brown,5.0072041267
8,152 · Wild Justice,0.0020656687,111 · Freckles,4.7088877853
9,130 · Orthodoxy,0.0013775373,22 · Roget's Thesaurus,4.6824969642


... (7 outputs left out)

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,108 · The Return of Sherlock Holmes,0.1342279341,108 · The Return of Sherlock Holmes,16.5254838383
2,103 · Around the World in Eighty Days,0.0109158866,102 · The Tragedy of Pudd'nhead Wilson,10.6093554647
3,204 · The innocence of Father Brown,0.0063255722,157 · Daddy-Long-Legs,9.9179589420
4,112 · Violists,0.0036370793,112 · Violists,8.9348801762
5,173 · The Insidious Dr. Fu Manchu,0.0032350107,101 · The Hacker Crackdown: Law and Disorder on the Electronic Frontier,6.2352413268
6,102 · The Tragedy of Pudd'nhead Wilson,0.0021891952,"200 · The Project Gutenberg Encyclopedia, Volume 1 of 28",6.1822037280
7,155 · The Moonstone,0.0019270560,138 · George Sand: Some Aspects of Her Life and Writings,6.1418942997
8,157 · Daddy-Long-Legs,0.0016512717,167 · American Hand Book of the Daguerreotype,5.7918833499
9,101 · The Hacker Crackdown: Law and Disorder on the Electronic Frontier,0.0016282954,155 · The Moonstone,5.4464687852


**Q8 · phileas fogg passepartout around the world**

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,103 · Around the World in Eighty Days,0.5656709851,103 · Around the World in Eighty Days,28.0839215007
2,21 · Three hundred Aesop’s fables,0.0769327238,119 · A Tramp Abroad,0.3597962056
3,19 · The Song of Hiawatha,0.0757196369,111 · Freckles,0.3595654871
4,"200 · The Project Gutenberg Encyclopedia, Volume 1 of 28",0.0740420641,125 · A Girl of the Limberlost,0.3589936658
5,164 · Twenty Thousand Leagues under the Sea,0.0722132146,118 · Big Dummy's Guide to the Internet,0.3582529982
6,136 · A Child's Garden of Verses,0.0717177016,162 · Take Me for a Ride: Coming of Age in a Destructive Cult,0.3581020587
7,18 · The Federalist Papers,0.0710010005,112 · Violists,0.3580692405
8,151 · The Rime of the Ancient Mariner,0.0699070698,152 · Wild Justice,0.3569314805
9,168 · The History and Practice of the Art of Photography,0.0692089902,101 · The Hacker Crackdown: Law and Disorder on the Electronic Frontier,0.3568484691


,Tamaño TF-IDF,Tamaño BM25,IDs en común,Primer ID TF-IDF,Primer ID BM25
Consulta,,,,,
Q1,10,10,6,21,200
Q2,2,2,2,13,12
Q3,10,10,6,11,11
Q4,10,10,6,16,120
Q5,10,10,5,161,161
Q6,10,10,4,1,14
Q7,10,10,7,108,108
Q8,10,10,1,103,103


Q2 recupera los mismos dos documentos con ambos métodos y los ordena de forma inversa. Q8 solo comparte el primer documento, ID 103, entre sus dos listas de diez resultados.

### Inversión de posiciones en Q4

Se comparan el ID 16 (*Peter Pan*) y el ID 120 (*Treasure Island*). Las posiciones se calculan sobre todos los documentos con puntuación positiva.

In [6]:
resultado_q4 = resultados["Q4"]
filas_q4 = [ids.index(doc_id) for doc_id in ("16", "120")]
display(pd.DataFrame([
    {"ID": ids[i], "Longitud": longitudes[i],
     "Posición TF-IDF": resultado_q4["orden_coseno"].index(i) + 1,
     "Coseno": resultado_q4["coseno"][i],
     "Posición BM25": resultado_q4["orden_bm25"].index(i) + 1,
     "Score BM25": resultado_q4["bm25"][i]}
    for i in filas_q4
]).set_index("ID").round(6))

terminos, frecuencias, aportes = aportes_bm25(tokens_consultas["Q4"])
indices = [columnas[t] for t in terminos]
vectores_d = matriz_tfidf[filas_q4]
vector_q = vector_tfidf(tokens_consultas["Q4"])
numeradores = vectores_d[:, indices].toarray() * vector_q[:, indices].toarray()
aportes_coseno = numeradores / (norma_dispersa(vectores_d, axis=1)[:, None] * norma_dispersa(vector_q))
assert np.allclose(aportes_coseno.sum(axis=1), resultado_q4["coseno"][filas_q4])

display(pd.DataFrame([
    {"ID": ids[i], "Longitud": longitudes[i], "Término": termino,
     "f(t,d)": frecuencias[i, j], "df(t)": df_por_termino[termino],
     "Aporte al coseno": aportes_coseno[fila, j], "Aporte BM25": aportes[i, j]}
    for fila, i in enumerate(filas_q4)
    for j, termino in enumerate(terminos)
]).set_index(["ID", "Término"]).round(6))

,Longitud,Posición TF-IDF,Coseno,Posición BM25,Score BM25
ID,,,,,
16,51199,1,0.026779,2,5.797241
120,73683,2,0.018740,1,5.917839


Longitud  f(t,d)  df(t)  Aporte al coseno  Aporte BM25
ID  Término                                                         
16  pirates      51199      56     19          0.021997     3.576822
    treasure     51199       3     53          0.000445     1.148941
    island       51199      35     61          0.004336     1.071478
120 pirates      73683      17     19          0.004548     3.454949
    treasure     73683      67     53          0.006767     1.381388
    island       73683      88     61          0.007425     1.081502

**TF-IDF con coseno.** El ID 16 ocupa el primer lugar con 0.026779, seguido del ID 120 con 0.018740. `pirates` es el término menos frecuente en la colección de los tres ($df=19$) y aparece 56 veces en el ID 16 frente a 17 en el ID 120. Su aporte al coseno es de 0.021997 frente a 0.004548, lo que inclina la comparación a favor del ID 16.

**BM25.** El ID 120 sube al primer puesto con 5.917839 y el ID 16 pasa al segundo con 5.797241. La longitud del ID 16 (51 199 tokens) favorece sus coincidencias frente al ID 120 (73 683 tokens). Aun con esa ventaja, la saturación comprime el efecto de las 56 frente a 17 apariciones de `pirates` a una diferencia de 0.122 en sus aportes. Los mayores aportes del ID 120 por `treasure` e `island` suman una ventaja de 0.243 y explican la inversión.

## 4. Efecto de los parámetros de BM25

Se utiliza Q7 en las tres ejecuciones. El ID 101 queda por debajo de la longitud media de 131 569.15 tokens y combina coincidencias de una y de 22 apariciones. El ID 200 supera la media y contiene los cuatro términos. El ID 108 sirve de referencia como ganador en las tres ejecuciones.

| Ejecución | $k_1$ | $b$ |
|---|---:|---:|
| Base | 1.2 | 0.75 |
| Sin ajuste por longitud | 1.2 | 0.00 |
| Mayor efecto de las repeticiones | 2.0 | 0.75 |

In [7]:
configuraciones = {"Base": (1.2, 0.75), "b = 0": (1.2, 0.0), "k₁ = 2": (2.0, 0.75)}
filas_q7 = [ids.index(doc_id) for doc_id in ("101", "200", "108")]
tokens_q7 = tokens_consultas["Q7"]
posiciones = [{"ID": ids[i], "Longitud": longitudes[i]} for i in filas_q7]
aportes_configuracion = {}
for nombre, (k1, b) in configuraciones.items():
    terminos, frecuencias, aportes = aportes_bm25(tokens_q7, k1, b)
    puntuaciones = aportes.sum(axis=1)
    orden = ordenar_positivos(puntuaciones, ids)
    aportes_configuracion[nombre] = aportes
    for registro, i in zip(posiciones, filas_q7):
        registro[f"Posición ({nombre})"] = orden.index(i) + 1
        registro[f"Score ({nombre})"] = puntuaciones[i]

display(pd.DataFrame(posiciones).set_index("ID").round(6))
display(pd.DataFrame([
    {"ID": ids[i], "Término": termino, "f(t,d)": frecuencias[i, j],
     **{nombre: aportes[i, j] for nombre, aportes in aportes_configuracion.items()}}
    for i in filas_q7
    for j, termino in enumerate(terminos)
]).set_index(["ID", "Término"]).round(6))

,Longitud,Posición (Base),Score (Base),Posición (b = 0),Score (b = 0),Posición (k₁ = 2),Score (k₁ = 2)
ID,,,,,,,
101,109250,5,6.235241,6,5.923079,8,6.928860
200,1363417,6,6.182204,2,11.198430,7,6.987341
108,117286,1,16.525484,1,16.495060,1,22.228853


f(t,d)      Base     b = 0    k₁ = 2
ID  Término                                            
101 sherlock            0  0.000000  0.000000  0.000000
    holmes              1  2.432559  2.263745  2.473549
    detective           1  1.883639  1.752920  1.915380
    investigation      22  1.919043  1.906415  2.539930
200 sherlock            1  0.567950  2.743318  0.482862
    holmes             22  3.464354  4.722641  3.927209
    detective           1  0.362907  1.752920  0.308538
    investigation      77  1.786993  1.979551  2.268731
108 sherlock           65  5.934658  5.925899  8.003737
    holmes            756  4.972989  4.972347  6.774772
    detective          22  3.672419  3.656953  4.853460
    investigation      33  1.945418  1.939860  2.596883

Con $b=0$ se elimina el ajuste por longitud. El ID 200 mide 1 363 417 tokens y sube del sexto al segundo puesto. Su score aumenta de 6.182204 a 11.198430. El ID 101, de 109 250 tokens, pierde la ventaja por su menor longitud y baja del quinto al sexto puesto.

Con $k_1=2$ y $b=0.75$ se necesita una frecuencia mayor para acercarse al máximo aporte de un término. En el ID 101, `investigation` (22 apariciones) aumenta de 1.919043 a 2.539930 y `detective` (una aparición) sube de 1.883639 a 1.915380. El score del ID 101 aumenta de 6.235241 a 6.928860, pero otros resultados ganan más puntuación y el documento baja del quinto al octavo lugar. El ID 108 conserva el primer lugar con 22.228853. Sus 65 apariciones de `sherlock` aportan 8.003737 con $k_1=2$ y la única aparición del ID 200 aporta 0.482862.